# Classification and Clustering

**Objectives:**
- Preprocess real-world data for machine learning (encoding, variable selection)
- Train classification models (Logistic Regression, K-Nearest Neighbors)
- Evaluate classifiers using accuracy and confusion matrices
- Compare models and validate on held-out data
- Apply K-Means clustering and relate clusters to labels

---

## What is Classification?

### Supervised Learning: Two Types of Prediction

When we build a predictive model, the type of output we want determines which approach to use:

| | **Regression** | **Classification** |
|:--|:--|:--|
| **Target variable** | Continuous number | Discrete category (label) |
| **Question asked** | "How much?" or "How many?" | "Which category?" |
| **Example** | Predict monthly salary: 42,300 | Predict credit score: "Good" |
| **Model output** | A number | A label |

In **classification**, what we predict is a **category** (also called a *label* or *class*), not a number.

### Real-World Examples of Classification

- **Spam detection** — is this email spam or not spam?
- **Medical diagnosis** — does this patient have diabetes?
- **Credit scoring** — is this client a Good, Standard, or Poor credit risk?
- **Image recognition** — is this photo a cat, a dog, or a bird?
- **Fraud detection** — is this transaction fraudulent?

### Our Task in This Notebook

We have records on thousands of bank clients. For each client, we know their financial details
(income, debt, payment habits, number of accounts...) and their **credit score category** —
`Good`, `Standard`, or `Poor`.

**Objective:** train a model that predicts a client's credit score category from their financial features.

### What You Will Learn Today

1. How to **prepare** data for a classification task (encoding, feature selection, train/test split)
2. How **Logistic Regression** works — and why it is called a "linear" classifier
3. How **K-Nearest Neighbors (KNN)** works — and why it is non-linear
4. How to **evaluate** a classifier beyond simple accuracy (confusion matrix, precision, recall)
5. How to **compare** two models and choose the best one using a held-out set
6. What **clustering** is and how it differs from classification

> **Reproducibility:** This notebook uses fixed random seeds (`random_state` parameters)
> wherever randomness is involved, so you will get **identical results** every time
> you run it from top to bottom.

## Part 1: Predicting the Credit Score

The two CSV files (source: [Kaggle](https://www.kaggle.com/datasets/clkmuhammed/creditscoreclassification?select=train.csv)) contain data about clients of a global finance company.

The goal is to predict a client's credit score category (**Good / Standard / Poor**) using available characteristics such as income, debt, payment behaviour, and other financial indicators.

This is a **classification problem**: instead of predicting a number, we predict a category.

To evaluate our models properly, we follow a **three-set approach**, which helps us measure how well our model performs on new, unseen data.

- `train.csv` is used to build the models.  
  It is further split into:
  - a **training set**, used to estimate the models
  - a **test set**, used to compare different models and choose the best one

- `test.csv` is kept completely separate and used only at the very end as a **validation set**, providing an unbiased final evaluation of the chosen model.

| dataset        | role |
|---------------|------|
| training set   | estimate model parameters |
| test set       | compare models and choose the best approach |
| validation set | final performance check on unseen data |

Using three datasets helps avoid choosing a model that performs well only by chance on a particular sample. It gives a more realistic idea of how the model will perform in practice.

---

Last week, we saw **k-fold cross-validation**, which is another way to evaluate models. Instead of splitting the data once into training and test sets, cross-validation repeatedly splits the data into different training and test subsets and averages performance across them. This gives a more stable comparison between models.

In this notebook, we use a simpler structure that separates clearly:
- learning the model
- comparing alternative methods
- evaluating final predictive performance

Other possible approaches include repeated train-test splits, k-fold cross-validation, or nested cross-validation. All aim to ensure that reported performance reflects how well the model generalizes to new data, not just how well it fits the sample used to estimate it.

---

### Step 1: Import the Data

We load two separate CSV files:
- **`train.csv`** — the main dataset used to build and evaluate our models
- **`test.csv`** — a completely separate file kept aside as a **validation set**

We do **not** look at the validation set until the very end.
Keeping it separate from the start ensures a truly unbiased final evaluation.

In [ ]:
import pandas
dataset = pandas.read_csv("train.csv")
validation = pandas.read_csv("test.csv")

### Step 2: Explore the Data

Before building any model, we explore the data to understand its structure:
- What columns (features) are available?
- What does the target variable look like?
- Are there any obvious issues (irrelevant columns, wrong data types)?

This step is always the first one — a model is only as good as the data it learns from.

In [ ]:
dataset.columns

# dataset.head()

### Step 3: Define and Encode the Target Variable



In [ ]:
dataset.head()

#### What is a target variable?

The **target variable** is the column we want to predict — the output of our model.
Here, the target is `Credit_Score`, which can take three text values:
`"Good"`, `"Standard"`, or `"Poor"`.

#### Why do we need to encode it?

**Problem:** Machine learning algorithms require numeric inputs and outputs.
They cannot work directly with raw text.

**Solution:** Use a `LabelEncoder` from scikit-learn to convert text categories into integers.
The encoder assigns numbers **in alphabetical order**, so the mapping is always consistent:

| Text label | Encoded value |
|:--|:--|
| `"Good"` | 0 |
| `"Poor"` | 1 |
| `"Standard"` | 2 |

We keep a record of this mapping using `inverse_transform`, so we can always convert
numbers back to meaningful labels when interpreting results.

> **Important:** After encoding, 0, 1, and 2 are just **category identifiers**, not real quantities.
> The model will not interpret 2 as "twice as much as 1".

In [ ]:
dataset['Credit_Score'].unique()

In [ ]:
from sklearn.preprocessing import LabelEncoder

cle = LabelEncoder()
dataset['Credit_Score'] = cle.fit_transform(dataset['Credit_Score'])


In [ ]:
# Check the mapping: which number corresponds to which label?
score_categories = cle.inverse_transform([0, 1, 2])
print("0 =", score_categories[0], "| 1 =", score_categories[1], "| 2 =", score_categories[2])

### Step 4: Encode Categorical Features

**Why?** Several features are stored as text (e.g., occupation, payment behaviour).
We apply the same encoding strategy: convert each text category into a number
so the model can use it.

We also drop the `Name` column — names are unique identifiers with no predictive value.
Including them would only add noise.

In [ ]:
#from sklearn.preprocessing import LabelEncoder as le

dataset['Payment_of_Min_Amount'] = cle.fit_transform(dataset['Payment_of_Min_Amount'])
dataset['Payment_Behaviour'] = cle.fit_transform(dataset['Payment_Behaviour'])
dataset['Occupation'] = cle.fit_transform(dataset['Occupation'])
dataset['Type_of_Loan'] = cle.fit_transform(dataset['Type_of_Loan'])
dataset['Credit_Mix'] = cle.fit_transform(dataset['Credit_Mix'])

# dataset.head()

In [ ]:
# Drop the Name column — it has no predictive value
dataset = dataset.drop(columns=["Name"], errors='ignore')

### Step 5: Visualize the Data

Plots help us understand distributions and relationships between variables before modelling.

In [ ]:
import seaborn as sns
sns.histplot(dataset['Credit_Score'])

> **Interpretation:** The three credit score categories are not perfectly balanced, but none is
> extremely rare. This is good — a heavily imbalanced dataset would require special handling.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(14,10))
sns.heatmap(dataset.corr(numeric_only=True))
# Uncomment the line below to see the heatmap with better color maps
#sns.heatmap(dataset.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.title("Correlation heatmap")
plt.show()

> **Interpretation:** ID, Customer_ID, SSN are not correlated with other variables (as expected — they are identifiers).
> `Occupation` also shows low correlation because LabelEncoder assigned arbitrary numbers to categories.
> A nonlinear model can still use it; for a linear model, one-hot encoding would be better.

---
### Step 6: Split Into Training and Test Sets

#### Why do we need a train/test split?

When a model trains on data, it learns from those specific examples. If we then evaluate
it on the **same data**, we get an overly optimistic picture — the model has essentially
"memorised" the training examples and may not generalise to new data.

To get an honest estimate of how the model performs on genuinely **new, unseen clients**,
we split the data into two parts:

| Set | Proportion | Purpose |
|:--|:--|:--|
| **Training set** | 75% | The model learns from this data |
| **Test set** | 25% | Used only to evaluate — never seen during training |

This mirrors the real world: we train on historical data, then apply the model to new clients.

> **`random_state=243`** — fixes the random split so you get the same training/test
> partition every time you run this cell.

In [ ]:
import sklearn.model_selection

df_train, df_test = sklearn.model_selection.train_test_split(
    dataset, test_size=0.25, random_state=243
)

### Step 7: Select Features and Prepare X / Y

**Why select variables?** Not all columns are useful features. IDs, SSNs, and similar identifiers
would add noise without any predictive value. We keep only meaningful financial predictors.

Scikit-learn convention:
- **`X`** — the feature matrix (input): the columns we use to predict
- **`Y`** — the target vector (output): the column we want to predict

In [ ]:
variables = [
    'Changed_Credit_Limit',
    'Payment_of_Min_Amount',
    'Credit_Mix',
    'Delay_from_due_date',
    'Annual_Income',
    'Monthly_Inhand_Salary',
    'Age',
    'Monthly_Balance',
    'Num_of_Delayed_Payment',
    'Outstanding_Debt',
    'Payment_Behaviour',
    'Credit_History_Age',
    'Num_Bank_Accounts',
    'Credit_Utilization_Ratio'
]

In [ ]:
# Features and labels for the training set
X = df_train[variables]
Y = df_train['Credit_Score']

# Features and labels for the test set
X_test = df_test[variables]
Y_test = df_test['Credit_Score']

---
## Step 8: Logistic Regression

### How Logistic Regression Works

#### Core idea

Logistic regression answers the question: **"What is the probability that this observation
belongs to a given class?"**

It starts like linear regression — computing a weighted sum of the input features:

```
score = w0 + w1 * income + w2 * debt + w3 * payment_history + ...
```

But instead of using this score directly as a prediction, it passes it through a mathematical
function (the **sigmoid function**) that squashes any number into a probability between 0 and 1.
If the probability is above 0.5, the model predicts that class.

#### Why is it called a "linear" classifier?

The **decision boundary** — the dividing line between classes — is always a **straight line**
(or flat surface in multiple dimensions). Logistic regression draws a straight line separating
"Good" from "not Good", and so on.

This makes logistic regression:
- **Fast** and easy to train
- **Interpretable** — each feature gets a coefficient that tells us its influence
- **Limited** when the true boundary between classes is curved or complex

#### Multi-class extension

Our problem has 3 classes (Good / Standard / Poor). Logistic regression handles this
internally by training 3 binary classifiers — one per class — and then predicting
the class with the highest probability.

Note: The model trains using an iterative algorithm. The default limit of 100 iterations is
often too few for complex datasets to converge (i.e., reach a stable solution).
Setting `max_iter=1000` gives the algorithm enough time to find the best parameters.

In [ ]:
from sklearn.linear_model import LogisticRegression

model_lr = LogisticRegression(max_iter=1000)
model_lr.fit(X, Y)

### Accuracy 

Accuracy = proportion of correctly predicted observations.

$$
\text{Accuracy} = \frac{\text{Number of correct predictions}}{\text{Total number of observations}}
$$

In sklearn classification models:

```python
model.score(X_test, Y_test)

In [ ]:
print("Training accuracy:", model_lr.score(X, Y))
print("Test accuracy:    ", model_lr.score(X_test, Y_test))

> **Interpretation:** Accuracy is the proportion of correctly classified observations.
> If training and test accuracy are close to each other, the model is not overfitting
> (it has not simply memorised the training data). If accuracy is modest overall,
> it may mean the classes are not linearly separable — a nonlinear model might do better.

---
## Step 9: Evaluating the Model — The Confusion Matrix

### Why Accuracy Alone Is Not Enough

Imagine a rare disease affects 1% of the population. A model that **always predicts "healthy"**
would be 99% accurate, yet it would never detect a single sick person.
Accuracy is a single number that hides *where* the model succeeds and where it fails.

### What the Confusion Matrix Shows

A confusion matrix compares **actual labels** (rows) against **predicted labels** (columns)
for every possible class:

| | Predicted: Good | Predicted: Standard | Predicted: Poor |
|:--|:--|:--|:--|
| **Actual: Good** | Correct prediction | Predicted Standard, was Good | Predicted Poor, was Good |
| **Actual: Standard** | Predicted Good, was Standard | Correct prediction | Predicted Poor, was Standard |
| **Actual: Poor** | Predicted Good, was Poor | Predicted Standard, was Poor | Correct prediction |

- **Diagonal values** (top-left to bottom-right) = **correct** predictions
- **Off-diagonal values** = **mistakes** — the model predicted the wrong class

The larger the diagonal values and the smaller the off-diagonal values, the better the model.



In [ ]:
actual = Y_test
predicted = model_lr.predict(X_test)

In [ ]:
from sklearn import metrics

confusion_matrix = metrics.confusion_matrix(actual, predicted)
confusion_matrix

In [ ]:
cm_display_lr = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_lr.plot(ax=ax)
plt.title("Logistic Regression — Confusion Matrix")
plt.show()

### Normalised Confusion Matrix

The raw confusion matrix shows **counts** (number of observations).   Normalising converts counts into **percentages**, making interpretation easier.

We can normalise in two ways:

1. **Row normalisation → Divide each row by its total  → Recall (Sensitivity)**

Among observations that truly belong to class X, what % did the model correctly identify?

2. *Column normalisation → Divide each column by its total  → Precision*

Among observations predicted as class X, what % actually belong to that class?

| Metric | How matrix is normalised | Key question |
|:--|:--|:--|
| **Recall** | rows sum to 1 | Did we correctly detect members of each class? |
| **Precision** | columns sum to 1 | When we predict a class, how often are we right? |

Both matter.  

In credit scoring:
- low recall → risky clients may not be detected
- low precision → good clients may be incorrectly flagged as risky

In [ ]:
# Normalize by row (recall): what % of each actual category was correctly detected?
cm_by_row = confusion_matrix / confusion_matrix.sum(axis=1)[:, None] * 100
cm_by_row

In [ ]:
cm_display_row = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm_by_row, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_row.plot(ax=ax)
plt.title("Logistic Regression — Normalised by Row (Recall)")
plt.show()

> **Interpretation (recall):** Look at the diagonal values. Each diagonal cell shows the
> percentage of clients in that category who were correctly identified. For example,
> if the "Poor" row shows 44% on the diagonal, it means only 44% of actual "Poor"
> clients were correctly detected — the other 56% were misclassified as Standard or Good.

In [ ]:
# Normalize by column (precision): of all predictions for a category, how many were correct?
cm_by_col = confusion_matrix / confusion_matrix.sum(axis=0)[None, :] * 100
cm_by_col

In [ ]:
cm_display_col = metrics.ConfusionMatrixDisplay(
    confusion_matrix=cm_by_col, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_col.plot(ax=ax)
plt.title("Logistic Regression — Normalised by Column (Precision)")
plt.show()

> **Interpretation (precision):** Each diagonal cell shows the percentage of *predicted*
> labels that were actually correct. For example, if the "Poor" column shows 65% on the
> diagonal, it means that of all clients the model flagged as "Poor", only 65% actually were —
> the other 35% were false alarms (clients wrongly classified as Poor).

---
## Step 10: K-Nearest Neighbors (KNN)

### How KNN Works

KNN is one of the most intuitive classifiers. To classify a new observation:

1. Find the **K most similar observations**
2. Take a **majority vote** of their labels
3. Assign the most common label to the new observation

Example: if K = 5 and most neighbours have a **Good** credit score → prediction = **Good**.

---

### What does "nearest" mean?

KNN measures similarity using **distance** in the feature space.
Closer observations are considered more similar.

Default in scikit-learn: **K = 5 neighbours**.

---

### Example with 1 feature

Suppose we only use **income**:

| Income | Credit score |
|-------|-------------|
| 2000 | Bad |
| 2500 | Good |
| 2700 | Good |

New client income = **2600**

Closest incomes → 2500 and 2700 → mostly **Good**.

---

### Example with 5 features

Similarity now depends on several characteristics:

| Client | Income | Age | Debt | Utilization | Late payments | Credit score |
|-------|--------|-----|------|-------------|--------------|-------------|
| A | 2000 | 25 | 5000 | 0.80 | 5 | Bad |
| B | 2500 | 30 | 2000 | 0.30 | 1 | Good |
| C | 2700 | 28 | 2500 | 0.40 | 2 | Good |

New client:

- Income = 2600
- Age = 29
- Debt = 2300
- Utilization = 0.35
- Late payments = 2

Distance is computed using all features together:

$$
d =
\sqrt{
(Income_i - Income_{new})^2
+ (Age_i - Age_{new})^2
+ (Debt_i - Debt_{new})^2
+ (Util_i - Util_{new})^2
+ (Late_i - Late_{new})^2
}
$$

KNN assigns the most common credit score among the K nearest neighbours.

Idea: similar financial profiles → similar credit scores.

#### Why KNN is non-linear

Unlike logistic regression, KNN does **not** draw a straight line between classes.
The decision boundary adapts to the actual shape of the data — it can be curved, irregular,
and complex. This makes KNN:
- **More flexible**: can capture patterns that a straight line cannot
- **More sensitive** to irrelevant features and noisy data
- **Slower** for large datasets (must compute distances to all training points)

#### Choosing K

| K value | Behaviour | Risk |
|:--|:--|:--|
| Very small (e.g., K=1) | Very sensitive to single points | Overfitting |
| Very large (e.g., K=100) | Very smooth boundary | Underfitting |
| Default K=5 | Good starting point | Balanced |

### A Note on Feature Scaling

Since KNN measures **distances**, the scale of each feature has a large impact.
If `Annual_Income` ranges from 10,000 to 200,000 while `Age` ranges from 18 to 80,
income will dominate the distance calculation — `Age` will barely influence predictions.

**Standard practice:** normalise the features (e.g., using `StandardScaler`) before
applying KNN, so each feature contributes equally to the distance.
In this notebook we skip scaling to compare models in a common baseline — but in practice
KNN can often be improved substantially with proper scaling.

### Logistic Regression vs KNN at a Glance

| | Logistic Regression | KNN |
|:--|:--|:--|
| **Type** | Linear | Non-linear |
| **Decision boundary** | Straight line | Flexible curve |
| **Handles complex patterns** | Limited | Better |
| **Sensitive to feature scale** | Less critical | Very important |
| **Interpretability** | High (coefficients) | Low |

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = KNeighborsClassifier()
model_knn.fit(X, Y)

In [ ]:
actual = Y_test
predicted = model_knn.predict(X_test)



In [ ]:
confusion_matrix_knn = metrics.confusion_matrix(actual, predicted)
cm_display_knn = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_knn, display_labels=score_categories
)

fig, ax = plt.subplots(figsize=(8, 8))
ax.grid(False)
cm_display_knn.plot(ax=ax)
plt.title("KNN Classifier")
plt.show()

#### Side-by-Side Comparison

Let's place both confusion matrices side by side to directly compare
how Logistic Regression and KNN perform on the same test set.

In [ ]:
from sklearn import metrics
import matplotlib.pyplot as plt

# Logistic Regression confusion matrix
predicted_lr = model_lr.predict(X_test)
confusion_matrix_lr = metrics.confusion_matrix(Y_test, predicted_lr)
cm_display_lr = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_lr,
    display_labels=score_categories
)

# KNN confusion matrix
predicted_knn = model_knn.predict(X_test)
confusion_matrix_knn = metrics.confusion_matrix(Y_test, predicted_knn)
cm_display_knn = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_knn,
    display_labels=score_categories
)

In [ ]:
# Side-by-side plot
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

ax = axes[0]
ax.grid(False)
cm_display_lr.plot(ax=ax)
ax.set_title("Logistic Regression")

ax = axes[1]
ax.grid(False)
cm_display_knn.plot(ax=ax)
ax.set_title("KNN Classifier")

plt.tight_layout()
plt.show()

> **Interpretation:** Compare the diagonal values of both matrices.
> Larger diagonal values mean more correct predictions.
> If KNN shows larger diagonal values than logistic regression, it suggests that
> the relationship between features and credit score has **nonlinear components**
> that KNN can capture but logistic regression — with its straight-line boundary — cannot.


As discussed above, raw confusion matrices show **counts**, but counts can be misleading when classes have very different sizes.

It is often more informative to look at **normalised confusion matrices**, which show **percentages** instead of counts.

Here we normalise **by row**, so each row sums to 1.  Remember that this allows us to evaluate **recall**: among observations truly belonging to a class, what proportion is correctly predicted?

This makes model comparison easier across classes.

In [ ]:
from sklearn import metrics
import matplotlib.pyplot as plt

# Logistic Regression confusion matrix
predicted_lr = model_lr.predict(X_test)
confusion_matrix_lr = metrics.confusion_matrix(Y_test, predicted_lr, normalize="true")
cm_display_lr = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_lr,
    display_labels=score_categories
)

# KNN confusion matrix
predicted_knn = model_knn.predict(X_test)
confusion_matrix_knn = metrics.confusion_matrix(Y_test, predicted_knn, normalize="true")
cm_display_knn = metrics.ConfusionMatrixDisplay(
    confusion_matrix=confusion_matrix_knn,
    display_labels=score_categories
)

# Side-by-side plot
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

ax = axes[0]
ax.grid(False)
cm_display_lr.plot(ax=ax)
ax.set_title("Logistic Regression (normalized)")

ax = axes[1]
ax.grid(False)
cm_display_knn.plot(ax=ax)
ax.set_title("KNN Classifier (normalized)")

plt.tight_layout()
plt.show()

---
### Step 11: Validate on the Held-Out Set

**Why?** We chose KNN because it performed better *on the test set*. But since the test
set influenced our model selection decision, it is no longer a truly independent evaluation.
The **validation set** (`test.csv`) — which we have never touched — gives us an unbiased
estimate of real-world performance.

We must preprocess the validation set in exactly the same way as the training data.

> **Note:** `LabelEncoder` assigns numbers in alphabetical order, so applying it separately
> to two datasets produces the same mapping — as long as both datasets contain the same categories.

In [ ]:
from sklearn.preprocessing import LabelEncoder

validation['Payment_of_Min_Amount'] = LabelEncoder().fit_transform(validation['Payment_of_Min_Amount'])
validation['Payment_Behaviour'] = LabelEncoder().fit_transform(validation['Payment_Behaviour'])
validation['Occupation'] = LabelEncoder().fit_transform(validation['Occupation'])
validation['Type_of_Loan'] = LabelEncoder().fit_transform(validation['Type_of_Loan'])
validation['Credit_Mix'] = LabelEncoder().fit_transform(validation['Credit_Mix'])
validation['Credit_Score'] = LabelEncoder().fit_transform(validation['Credit_Score'])

In [ ]:
X_valid = validation[variables]
Y_valid = validation['Credit_Score']

In [ ]:
actual_valid = Y_valid
predicted_valid = model_knn.predict(X_valid)

In [ ]:
confusion_matrix_valid = metrics.confusion_matrix(actual_valid, predicted_valid)

fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(14, 5))

ax = axes[0]
cm = confusion_matrix_valid / confusion_matrix_valid.sum(axis=0)[None, :]
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=score_categories)
cm_display.plot(ax=ax)
ax.set_title("Normalised by Column (Precision)")

ax = axes[1]
cm = confusion_matrix_valid / confusion_matrix_valid.sum(axis=1)[:, None]
cm_display = metrics.ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=score_categories)
cm_display.plot(ax=ax)
ax.set_title("Normalised by Row (Recall)")

plt.tight_layout()
plt.show()

> **Interpretation:** Compare validation results to the test set results.
> If the diagonal values are similar in both cases, the model generalises well —
> it performs consistently on data it has never seen before.
> A large gap between test and validation performance would be a warning sign
> of overfitting or inconsistent data preprocessing.

---
## Part 2: Segmenting the Bank Clients (K-Means Clustering)

### Classification vs. Clustering: What is the Difference?

| | **Classification (supervised)** | **Clustering (unsupervised)** |
|:--|:--|:--|
| **Labels available?** | Yes — we know the target | No — we discover groups from the data |
| **Goal** | Predict a known category | Find natural groups |
| **Example** | Predict if a client is "Good" | Find groups of similar clients |

In **clustering**, we give the model no labels at all. It discovers groups entirely on its own,
based purely on the similarity between observations.

### Our Clustering Goal

Can we find natural groups among clients **without using the credit score**?
And if so, do these natural groups correspond to credit quality categories?

We use **K-Means clustering**, which tries to partition observations into K groups
(here K=3, matching our 3 credit categories) by minimising the distance of each point
to its cluster centre.

> **`random_state=42`** — K-Means uses a random initialisation step.
> Setting a fixed seed ensures the same clusters are found every run.

In [ ]:
from sklearn.cluster import KMeans

km_model = KMeans(n_clusters=3, random_state=42)
km_model.fit(dataset.select_dtypes(include='number'))

In [ ]:
# Assign each client to a cluster
dataset['cluster'] = km_model.predict(dataset.select_dtypes(include='number'))

### Are the Clusters Related to the Credit Score?

Let's check whether the three clusters discovered by K-Means correspond to the three
credit score categories. If they do, it means creditworthiness is the main axis
along which clients differ.

In [ ]:
dataset.groupby('cluster')['Credit_Score'].value_counts(normalize=True)

> **Interpretation:** If the proportion of each credit score category is **similar across clusters**,
> it means the clusters are **not** aligned with creditworthiness.
> The algorithm found other dimensions of client similarity (e.g., income level or spending behaviour)
> that do not strongly correspond to the ability to repay debt.
>
> This is an important lesson: just because two clients belong to the same cluster
> does not mean they have the same credit risk.

## From prediction to clustering

So far, we used supervised learning methods to **predict credit score**.

Banks, however, are often interested in a different question:
can we identify **types of clients** with similar financial situations?

Such groups can help banks design:
- tailored financial products
- adapted credit limits
- personalised repayment plans
- better risk monitoring strategies

To illustrate this idea, we focus on two simple characteristics:

- annual income → financial capacity
- credit utilization ratio → intensity of credit use

Together, these variables describe how much clients earn and how strongly they rely on credit.



In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

# choose two variables
X = dataset[["Annual_Income", "Credit_Utilization_Ratio"]]

# standardize before K-means
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# fit K-means
km_model = KMeans(n_clusters=3, random_state=42)
dataset["cluster"] = km_model.fit_predict(X_scaled)



Using only two variables allows us to visualise the clusters directly.
Each point in the graph represents one client, and colours indicate
different financial profiles discovered by the algorithm.

In [ ]:
# scatterplot in original units
plt.figure(figsize=(8, 6))
plt.scatter(
    dataset["Annual_Income"],
    dataset["Credit_Utilization_Ratio"],
    c=dataset["cluster"],
    alpha=0.6
)

plt.xlabel("Annual Income")
plt.ylabel("Credit Utilization Ratio")
plt.title("Client groups based on income and credit usage")
plt.show()

### Interpretation of the clusters

The clustering reveals three broad financial profiles:

• lower income – lower credit utilization  
• lower income – higher credit utilization  
• higher income clients with more diverse credit usage patterns  

The algorithm separates clients along two important dimensions:

- financial capacity (income)
- intensity of reliance on credit (utilization ratio)

This type of segmentation can be useful for a bank because different groups of clients may require different financial strategies.

For example:

- clients with lower income and high utilization may require closer risk monitoring
- higher income clients may be eligible for larger credit limits or premium products
- clients with moderate utilization may represent stable borrowing behaviour

Clustering therefore helps identify **types of financial behaviour**, even without explicitly predicting credit risk.

This illustrates how unsupervised learning can support decision-making by revealing structure in the data.